# CSE 144 Final Project — 100-Class Image Classification

In [21]:
# Imports
# !pip -q install torch torchvision matplotlib tqdm scikit-learn pillow pandas

# file system
import os
import glob
# random.seed
import random
# arrays, csv
import numpy as np
import pandas as pd
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
# Feeding image to model in batches
from torch.utils.data import Dataset, DataLoader
# Resizing, cropping, normalizing images before model training
from torchvision import transforms
# ImageNet pretrained model
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
# .jpg
from PIL import Image
# Progress bars during training models
from tqdm.auto import tqdm
# Matplotlib
import matplotlib.pyplot as plt
# train_test_split
from sklearn.model_selection import train_test_split

In [22]:
# Device & Seed

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("device:", device)

device: mps


In [32]:
# Config

DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_CLASSES = 100
# IMG_SIZE = 300 # This is ImageNet's recommendation
IMG_SIZE = 224 # This is professor's suggestion
BATCH_SIZE = 32 # tune this
NUM_WORKERS = 0
EPOCHS_HEAD = 5 # tune this
EPOCHS_FINETUNE = 20 # tune this
CKPT_PATH = os.path.join(CKPT_DIR, "best_effb3.pt")

# https://pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b3.html
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [24]:
# https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset
# https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.convert
class LabeledDataset(Dataset):
    # Store list of (path, label) pairs and the transform
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    # Returns images length for end of epoch in PyTorch
    def __len__(self):
        return len(self.samples)

    # Given an index, open image, convert RGB, apply the transform, return image + label
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [25]:
# https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset
# https://docs.python.org/3/library/glob.html
# https://docs.python.org/3/library/os.path.htm
# https://docs.python.org/3/howto/sorting.html
class TestDataset(Dataset):
    # Return filename (ID in submission CSV) instead of label
    def __init__(self, test_dir, transform=None):
        self.paths = sorted(
            glob.glob(os.path.join(test_dir, "*.jpg")),
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
        )
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

In [26]:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
all_paths, all_labels = [], []
for class_id in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_id))
    for fname in sorted(os.listdir(class_dir)):
        if fname.endswith(".jpg"):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

tr_paths, v1_paths, tr_labels, v1_labels = train_test_split(all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42)

train_samples = list(zip(tr_paths, tr_labels))
val_samples = list(zip(v1_paths, v1_labels))

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

Train: 863, Val: 216


In [27]:
# https://pytorch.org/vision/stable/transforms.html
# https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = LabeledDataset(train_samples, transform=train_tf)
val_ds = LabeledDataset(val_samples, transform=val_tf)
test_ds = TestDataset(TEST_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

Batches — train: 27, val: 7, test: 33


In [28]:
# Model
# https://pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b3.html

def build_model(num_classes=NUM_CLASSES, freeze_backbone=True):
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.clsasifer = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes),
    )
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    return model

model = build_model(freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

Trainable: 1,690,700 / 12,386,932 params


In [29]:
# Loss function

criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # 0.1 default

def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n += imgs.size(0)
    if scheduler:
        scheduler.step()
    return total_loss / n, total_correct / n

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        total_correct += (out.argmax(1) == labels).sum().item()
        n += imgs.size(0)
    return total_loss / n, total_correct / n

In [30]:
# Phase 1 training (head only)

# AdamW — https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html
# CosineAnnealingLR — https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html
# torch.save — https://pytorch.org/docs/stable/generated/torch.save.html

optimizer_head = optim.AdamW(
  filter(lambda p: p.requires_grad, model.parameters()),
  lr=1e-3, weight_decay=1e-4
)
scheduler_head = optim.lr_scheduler.CosineAnnealingLR(optimizer_head, T_max=EPOCHS_HEAD)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

print("=== Phase 1: Head only ===")
for epoch in range(EPOCHS_HEAD):
  tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_head, scheduler_head)
  vl_loss, vl_acc = evaluate(model, val_loader)
  for k, v in zip(["train_loss","train_acc","val_loss","val_acc"],
                  [tr_loss, tr_acc, vl_loss, vl_acc]):
      history[k].append(v)
  if vl_acc > best_val_acc:
      best_val_acc = vl_acc
      torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_acc": vl_acc}, CKPT_PATH)
  print(f"  [{epoch+1}/{EPOCHS_HEAD}] train {tr_acc:.4f} | val {vl_acc:.4f}")

print(f"\nBest so far: {best_val_acc:.4f}")

=== Phase 1: Head only ===


  0%|          | 0/27 [00:00<?, ?it/s]

  [1/5] train 0.0023 | val 0.0046


  0%|          | 0/27 [00:00<?, ?it/s]

  [2/5] train 0.0197 | val 0.0231


...

  [3/5] train 0.0753 | val 0.0417


  0%|          | 0/27 [00:00<?, ?it/s]

  [4/5] train 0.1043 | val 0.0741


  0%|          | 0/27 [00:00<?, ?it/s]

  [5/5] train 0.1298 | val 0.0787

Best so far: 0.0787


In [33]:
# Phase 2 (full fine-tuning)

for param in model.parameters():
  param.requires_grad = True

optimizer_ft = optim.AdamW([
  {"params": model.features.parameters(), "lr": 1e-5},
  {"params": model.classifier.parameters(), "lr": 1e-4},
], weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FINETUNE)

print("=== Phase 2: Full fine-tuning ===")
for epoch in range(EPOCHS_FINETUNE):
  tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_ft, scheduler_ft)
  vl_loss, vl_acc = evaluate(model, val_loader)
  for k, v in zip(["train_loss","train_acc","val_loss","val_acc"],
                  [tr_loss, tr_acc, vl_loss, vl_acc]):
      history[k].append(v)
  if vl_acc > best_val_acc:
      best_val_acc = vl_acc
      torch.save({"model_state_dict": model.state_dict(), "epoch": EPOCHS_HEAD + epoch, "val_acc": vl_acc},
CKPT_PATH)
  print(f"  [{epoch+1}/{EPOCHS_FINETUNE}] train {tr_acc:.4f} | val {vl_acc:.4f}")

print(f"\nBest val acc: {best_val_acc:.4f}")

=== Phase 2: Full fine-tuning ===


  0%|          | 0/27 [00:00<?, ?it/s]

  [1/20] train 0.1472 | val 0.0880


  0%|          | 0/27 [00:00<?, ?it/s]

  [2/20] train 0.1599 | val 0.0926


...

  [18/20] train 0.3673 | val 0.2685


  0%|          | 0/27 [00:00<?, ?it/s]

  [19/20] train 0.3534 | val 0.2593


  0%|          | 0/27 [00:00<?, ?it/s]

  [20/20] train 0.3778 | val 0.2546

Best val acc: 0.2685
